# Sentiment Classification — RNN vs GRU vs LSTM

This notebook mirrors the character-level model comparison, but the task is different.

Task:
- input: a short review sentence
- output: a sentiment label (`positive` or `negative`)

We will train three sequence classifiers:
- Simple RNN
- GRU
- LSTM

How to read the code:
- first we build the word vocabulary and dataset loaders
- then we define one configurable recurrent classifier
- finally we train all three architectures under the same settings and compare them fairly

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "nirma_university_language_models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt

from nirma_university_language_models import (
    SentimentRNNClassifier,
    build_sentiment_dataloaders,
    build_word_vocabulary,
    get_device,
    label_distribution,
    load_sentiment_examples,
    predict_sentiment,
    seed_everything,
    train_sentiment_model,
)


In [ ]:
device = get_device()
SEED = 42
seed_everything(SEED)

print("Using device:", device)
print("Seed:", SEED)

## 1. Load the labeled review data

The next code cell loads the local CSV dataset, extracts the review texts, and builds the word vocabulary.

This gives us:
- the raw examples
- the sentiment labels
- a shared vocabulary for all three models

In [ ]:
examples, DATA_PATH = load_sentiment_examples()
texts = [example["text"] for example in examples]
vocab, token_to_idx, idx_to_token = build_word_vocabulary(texts)

print("Dataset path:", DATA_PATH)
print("Number of examples:", len(examples))
print("Label distribution:", label_distribution(examples))
print("Vocabulary size:", len(vocab))

## 2. Build training and validation loaders

A sentiment classifier reads a full sequence and predicts one class label.

In the next code cell:
- `MAX_LEN` controls how many word positions we keep per review
- `BATCH_SIZE` controls how many reviews are processed at once
- the helper converts text into padded integer tensors and creates train/validation splits

In [ ]:
MAX_LEN = 12
BATCH_SIZE = 8
train_loader, val_loader = build_sentiment_dataloaders(
    examples,
    token_to_idx,
    max_len=MAX_LEN,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

print("Training examples:", len(train_loader.dataset))
print("Validation examples:", len(val_loader.dataset))

## 3. Define the recurrent classifier configuration

The class itself lives in the shared helper module, but the notebook still controls the main hyperparameters.

What the next code cell shows:
- the embedding size for each word token
- the hidden-state size used by the recurrent layer
- dropout before the final classification layer
- a preview of the RNN-based classifier architecture

In [ ]:
MODEL_KWARGS = {"embed_dim": 64, "hidden_size": 64, "dropout": 0.2}
preview_model = SentimentRNNClassifier(len(vocab), rnn_type="RNN", **MODEL_KWARGS)
preview_model

## 4. Training helper

The training loop is wrapped in `train_sentiment_model(...)` so the notebook can stay readable.

Conceptually that helper does the usual deep-learning steps:
- run a forward pass on a batch of padded review sequences
- compute cross-entropy loss against the true sentiment labels
- backpropagate the gradients
- update the model parameters with Adam
- track both loss and accuracy on training and validation data

In [ ]:
print("Model config:", MODEL_KWARGS)
print("Classification task: negative = 0, positive = 1")

## 5. Train RNN, GRU, and LSTM under the same settings

This is the key comparison section.

The next code cell trains all three models using the same:
- dataset split
- maximum sequence length
- batch size
- embedding size
- hidden size
- learning rate
- epoch count

That means any major performance difference is mainly due to the recurrent architecture.

In [ ]:
EPOCHS = 8

rnn_model, rnn_history = train_sentiment_model(
    "RNN",
    len(vocab),
    train_loader,
    val_loader,
    device,
    epochs=EPOCHS,
    **MODEL_KWARGS,
)
gru_model, gru_history = train_sentiment_model(
    "GRU",
    len(vocab),
    train_loader,
    val_loader,
    device,
    epochs=EPOCHS,
    **MODEL_KWARGS,
)
lstm_model, lstm_history = train_sentiment_model(
    "LSTM",
    len(vocab),
    train_loader,
    val_loader,
    device,
    epochs=EPOCHS,
    **MODEL_KWARGS,
)

## 6. Compare loss curves

Loss shows how confidently each model matches the correct labels during training and validation.

When students look at the next plot they should ask:
- which model learns faster?
- which model reaches the lowest validation loss?
- do any curves suggest overfitting?

In [ ]:
epochs = list(range(1, EPOCHS + 1))

plt.figure(figsize=(10, 5))
plt.plot(epochs, rnn_history["train_loss"], marker="o", label="RNN Train")
plt.plot(epochs, rnn_history["val_loss"], marker="o", label="RNN Val")
plt.plot(epochs, gru_history["train_loss"], marker="o", label="GRU Train")
plt.plot(epochs, gru_history["val_loss"], marker="o", label="GRU Val")
plt.plot(epochs, lstm_history["train_loss"], marker="o", label="LSTM Train")
plt.plot(epochs, lstm_history["val_loss"], marker="o", label="LSTM Val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sentiment Classification Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Compare accuracy curves

Accuracy is often easier for students to interpret than loss because it directly measures the fraction of correct predictions.

The next plot compares training and validation accuracy across the three recurrent models.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(epochs, rnn_history["train_acc"], marker="o", label="RNN Train")
plt.plot(epochs, rnn_history["val_acc"], marker="o", label="RNN Val")
plt.plot(epochs, gru_history["train_acc"], marker="o", label="GRU Train")
plt.plot(epochs, gru_history["val_acc"], marker="o", label="GRU Val")
plt.plot(epochs, lstm_history["train_acc"], marker="o", label="LSTM Train")
plt.plot(epochs, lstm_history["val_acc"], marker="o", label="LSTM Val")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Sentiment Classification Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Predict sentiment for new sentences

The final step is to use the trained models on fresh input sentences.

This helps students connect the numeric training metrics to actual model behavior.

In [ ]:
test_sentences = [
    "the movie was heartfelt and surprisingly funny",
    "this film was slow predictable and boring",
    "i liked the acting but the ending was weak",
    "the story was sharp moving and satisfying",
]

models = {
    "RNN": rnn_model,
    "GRU": gru_model,
    "LSTM": lstm_model,
}

for sentence in test_sentences:
    print(f"Sentence: {sentence}")
    for model_name, model in models.items():
        label, probabilities = predict_sentiment(
            model,
            sentence,
            token_to_idx,
            device,
            max_len=MAX_LEN,
        )
        print(f"  {model_name}: {label} | probs={probabilities}")
    print("-" * 100)